In [1]:
import numpy as np
import pandas as pd

In [2]:
links = pd.read_csv("links.csv")
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")

In [3]:
links

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


In [4]:
movies


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [5]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [6]:
tags

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


In [7]:
tags = tags.groupby('movieId')['tag'].apply(lambda x: " ".join(x)).reset_index()



In [8]:
movies = movies.merge(tags, on='movieId', how='left')

In [9]:
movies.head()

,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,NaN
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake


In [10]:
movies.isnull().sum()

movieId       0
title         0
genres        0
tag        8170
dtype: int64

In [15]:
movies['tag'] = movies['tag'].fillna('')


In [16]:
movies['genres'] = movies['genres'].apply(lambda x:x.split('|'))

In [18]:
movies['tag'] = movies['tag'].apply(lambda x: str(x).split())

In [19]:
movies.head()

,movieId,title,genres,tag
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]","[pixar, pixar, fun]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]","[fantasy, magic, board, game, Robin, Williams,..."
2,3,Grumpier Old Men (1995),"[Comedy, Romance]","[moldy, old]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",[]
4,5,Father of the Bride Part II (1995),[Comedy],"[pregnancy, remake]"


In [20]:
movies['tags'] = movies['genres'] + movies['tag']

In [21]:
new_df = movies[['movieId','title','tags']]

In [22]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

C:\Users\dell\anaconda3\envs\packt-py36\lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [23]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\dell\anaconda3\envs\packt-py36\lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [24]:
from sklearn.feature_extraction.text import CountVectorizer 
cv = CountVectorizer(max_features= 5000, stop_words='english')

In [28]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [29]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\dell\anaconda3\envs\packt-py36\lib\site-packages\ipykernel_launcher.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if sys.path[0] == '':


In [30]:
from sklearn.metrics.pairwise import cosine_similarity

content_similarity = cosine_similarity(vectors)


In [31]:
movie_ratings = ratings.pivot_table(index='movieId',columns='userId',values='rating').fillna(0)

In [32]:
collab_similarity = cosine_similarity(movie_ratings)

In [33]:
collab_similarity = pd.DataFrame(collab_similarity,
                                index=movie_ratings.index,
                                columns=movie_ratings.index)

In [38]:
def recommend(movie):

    movie = movie.lower()

    if movie not in new_df['title'].str.lower().values:
        print("Movie not found")
        return

    movie_index = new_df[new_df['title'].str.lower() == movie].index[0]
    movie_id = new_df.iloc[movie_index].movieId

    # Content similarity
    content_scores = list(enumerate(content_similarity[movie_index]))

    # Collaborative similarity (select only movies present in content-based list)
    collab_sim = collab_similarity[movie_id]
    collab_scores = []

    for i, row in new_df.iterrows():
        mid = row.movieId
        if mid in collab_sim.index:
            collab_scores.append(collab_sim.loc[mid])
        else:
            collab_scores.append(0)  # if missing, assign 0

    # Combine scores
    hybrid_scores = [(i, (content_scores[i][1] + collab_scores[i]) / 2) for i in range(len(content_scores))]

    # Sort top 5
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)[1:6]

    for i in hybrid_scores:
        print(new_df.iloc[i[0]].title)

In [39]:
recommend("Toy Story (1995)")

Bug's Life, A (1998)
Toy Story 2 (1999)
Monsters, Inc. (2001)
Shrek (2001)
Antz (1998)
